# Day 51 — Advanced feature engineering: target encoding & leakage prevention
Objectives:
- Understand target/mean encoding for high-cardinality categoricals.
- Prevent leakage with KFold schemes.
- Integrate into sklearn Pipelines.

In [ ]:
import pandas as pd, numpy as np, seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
df = sns.load_dataset('titanic').dropna(subset=['survived','sex','class','embarked','fare','age'])
X = df[['sex','class','embarked','fare','age']]
y = df['survived']
Xtr,Xte,ytr,yte = train_test_split(X,y, stratify=y, random_state=42)


## KFold target encoding utility (no leakage)
For each fold, compute means on train folds and apply to val fold only.

In [ ]:
def kfold_target_encode(cat: pd.Series, y: pd.Series, n_splits=5, smoothing=10):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    out = pd.Series(index=cat.index, dtype=float)
    global_mean = y.mean()
    for tidx, vidx in skf.split(cat, y):
        trc, trY = cat.iloc[tidx], y.iloc[tidx]
        means = trY.groupby(trc).mean()
        counts = trY.groupby(trc).size()
        smooth = (means * counts + global_mean * smoothing) / (counts + smoothing)
        out.iloc[vidx] = cat.iloc[vidx].map(smooth).fillna(global_mean)
    return out.fillna(global_mean)

Xe = Xtr.copy()
for col in ['sex','class','embarked']:
    Xe[col + '_te'] = kfold_target_encode(Xtr[col], ytr)
Xe[['sex_te','class_te','embarked_te']].head()


## Compare baseline One-Hot vs Target Encoding features
(Demonstration: combine OHE for small cats + TE for high-cardinality if present.)

In [ ]:
from sklearn.metrics import roc_auc_score
# Baseline OHE
ohe = ColumnTransformer([('ohe', OneHotEncoder(handle_unknown='ignore'), ['sex','class','embarked'])], remainder='passthrough')
pipe_ohe = Pipeline([('pre', ohe), ('clf', LogisticRegression(max_iter=1000))])
pipe_ohe.fit(Xtr,ytr); auc_ohe = roc_auc_score(yte, pipe_ohe.predict_proba(Xte)[:,1])
# TE approach
Xtr_te = Xtr.copy(); Xte_te = Xte.copy()
for c in ['sex','class','embarked']:
    Xtr_te[c+'_te'] = kfold_target_encode(Xtr[c], ytr)
    # map train means to test
    m = pd.concat([Xtr[c], ytr], axis=1).groupby(c)['survived'].mean()
    Xte_te[c+'_te'] = Xte[c].map(m).fillna(ytr.mean())
cols = ['fare','age','sex_te','class_te','embarked_te']
clf = LogisticRegression(max_iter=1000).fit(Xtr_te[cols], ytr)
auc_te = roc_auc_score(yte, clf.predict_proba(Xte_te[cols])[:,1])
auc_ohe, auc_te


## Exercises
1) Add KFold target encoding to a Pipeline via FunctionTransformer or custom Transformer.
2) Add smoothing and prior appropriately; experiment with n_splits.
3) Measure impact on ROC AUC vs OHE across multiple random splits.